In [ ]:
import base64
import requests
import time
import pandas as pd
import os

# OpenAI API Key - Replace with your GPT API key
api_key = ""

# Function to encode the image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# Read CSV
questions_df = pd.read_csv("C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/VLAT Questions.csv")
questions = questions_df.to_dict('records')
responses_data = []


for i, question in enumerate(questions, start=1):
    time.sleep(5)
    try:
        print(f"\nProcessing question {i}:")
        print(question)
        
        # Path to your image
        image_path = "C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/" + str(question.get('vis', '')) + ".png"
        
        # Question text
        question_text = question.get('question: ', '')
        question_options = question.get('option:', '')
        correct_ans = str(question.get('correct', '')).strip()
        
        print(f"Processing image: {image_path}")
        print(f"Question: {question_text}")
        print(f"Options: {question_options}")
        print(f"Correct answer: {correct_ans}")
        
        # Getting the base64 string
        base64_image = encode_image(image_path)
        
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}"
        }
        
        payload = {
            "model": "gpt-4.5-preview-2025-02-27", # Use GPT-4o for vision capabilities
            "max_tokens": 5000,
            "temperature": 0.0,
            "messages": [
                {
                    "role": "system",
                    "content": "You are an assistant, skilled in reading and interpreting visually represented data."
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": 'I am about to show you a graph and ask you a multiple-choice question about that graph. \n\n' +
                                   'Task 1: Data Extraction and Table Creation: First, explicitly list ALL numerical values you can identify on both axes, then create a structured table using markdown syntax that includes ALL data points you identified above with appropriate column headers with units. \n \n' +
                                   'Task 2: Sort the data: Sort the data in descending order by the numerical values. \n \n' +
                                   'Task 3: Data Verification and Error Handling: Double-check if your table matches ALL elements in the graph by comparing each value in your table with the graph and updating your table with correct values, verify the sorting is correct, and before proceeding, confirm all corrections have been made and use ONLY the corrected data for analysis. \n \n' +
                                   'Task 4: Question Analysis: Using ONLY the verified data in your table, compare EACH value individually with the reference value, for "less than" comparisons mark ALL values that are even slightly below the reference, for "greater than" comparisons mark ALL values that are even slightly above the reference, and show each comparison on a new line. \n \n' +
                                   'Provide your reasoning with specific references to table values. \n\n' +
                                   'End with: "Correct Answer: ". Just write the value, nothing else. Do not write anything after this. \n\n' +
                                   'Let\'s solve this step by step.' +
                                   '\n\n' + question_text + " " + question_options
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/png;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ]
        }
        
        # Add retry logic for overloaded errors
        max_retries = 3
        retry_delay = 20  # seconds
        
        for retry in range(max_retries):
            try:
                time_start = time.perf_counter()
                response_raw = requests.post(
                    "https://api.openai.com/v1/chat/completions",
                    headers=headers,
                    json=payload
                )
                time_end = time.perf_counter()
                
                response = response_raw.json()
                
                # If we get a rate limit error, wait and retry
                if 'error' in response and response['error'].get('code') == 'rate_limit_exceeded':
                    if retry < max_retries - 1:  # Don't sleep on the last retry
                        print(f"\nAPI rate limited, waiting {retry_delay} seconds before retry {retry + 1}/{max_retries}")
                        time.sleep(retry_delay)
                        continue
                
                break  # If we get here, we either got a good response or a different error
                
            except Exception as e:
                print(f"Error during API call (attempt {retry + 1}/{max_retries}):", str(e))
                if retry < max_retries - 1:
                    time.sleep(retry_delay)
                    continue
                raise  # Re-raise the last exception if we've exhausted all retries
        print("\nAPI Response:", response)  # Debug print
        
        if 'error' in response:
            gpt_answer = f"Error: {response['error'].get('message', 'Unknown error')}"
            is_correct = "N/A"
        else:
            # Extract the answer after "Correct Answer: "
            full_response = response["choices"][0]["message"]["content"].strip()
            if "Correct Answer: " in full_response:
                gpt_answer = full_response.split("Correct Answer: ")[-1].strip()
                # Case-insensitive comparison after stripping whitespace
                is_correct = gpt_answer.strip().upper() == correct_ans.strip().upper()
            else:
                gpt_answer = "Error: No answer in correct format"
                is_correct = "N/A"
        
        responses_data.append([gpt_answer, time_end-time_start, is_correct])
        print(f"\nAnswer: {gpt_answer}")
        print(f"Time taken: {time_end-time_start:.2f} seconds")
        print(f"Correct? {is_correct}")
        
        # Increase delay between requests to 15 seconds
        time.sleep(max(15 - (time_end-time_start), 0))
        
    except Exception as e:
        print(f"Error processing question {i}:", str(e))
        responses_data.append([f"Error: {str(e)}", 0, "N/A"])

# Create Results directory if it doesn't exist
results_dir = "C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Results/2_GPT 4.5_Final Results with Explanation/"
os.makedirs(results_dir, exist_ok=True)

results_df = pd.DataFrame(responses_data, columns=['response', 'time', 'correct_bool'])
results_df.index = range(1, results_df.shape[0] + 1)
results_df.to_csv(results_dir + "GPT_VLAT_" + str(int(time.time())) + ".csv", index_label="id")
print("\n*** Finished ***")


Processing question 1:
{'id': 1, 'dropped': 'no', 'vis': 'VLAT_a', 'item': 'a_1', 'question: ': 'What was the price of a barrel of oil in February 2015? ', 'option:': '$39.72; $57.36; $48.90; $62.85', 'correct': '$48.90 '}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_a.png
Question: What was the price of a barrel of oil in February 2015? 
Options: $39.72; $57.36; $48.90; $62.85
Correct answer: $48.90

API Response: {'id': 'chatcmpl-B9M2VALlU1LPqbqUP8JdYEVMLxr0u', 'object': 'chat.completion', 'created': 1741569371, 'model': 'gpt-4.5-preview-2025-02-27', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Task 1: Data Extraction and Table Creation\n\nNumerical values identified on axes:\n\nY-axis (Oil Price in $ per barrel):\n65, 60, 55, 50, 45, 40, 35\n\nX-axis (Months of 2015):\nJanuary, February, March, April, May, June, July, August, September, October, November, December\n\nExtracted Data Points (approximate):\n- Jan


Processing question 4:
{'id': 4, 'dropped': 'no', 'vis': 'VLAT_a', 'item': 'a_4', 'question: ': 'Over the course of the second half of 2015, the price of a barrel of oil was ____________.', 'option:': 'rising; falling; staying', 'correct': 'rising'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_a.png
Question: Over the course of the second half of 2015, the price of a barrel of oil was ____________.
Options: rising; falling; staying
Correct answer: rising

API Response: {'id': 'chatcmpl-B9M6y8z87B3i4oaJSzZYICGpUcozd', 'object': 'chat.completion', 'created': 1741569648, 'model': 'gpt-4.5-preview-2025-02-27', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': "Task 1: Data Extraction and Table Creation\n\nNumerical values identified:\n\nY-axis (Oil Price in $ per barrel):\n- 35, 40, 45, 50, 55, 60, 65\n\nX-axis (Months of 2015):\n- January, February, March, April, May, June, July, August, September, October, November, Dece


API Response: {'id': 'chatcmpl-B9M9iCwpYghcJEEaXJS0bX0RCrWiX', 'object': 'chat.completion', 'created': 1741569818, 'model': 'gpt-4.5-preview-2025-02-27', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Task 1: Data Extraction and Table Creation\n\nNumerical values on axes:\nY-axis (Speed in Mbps): 0, 5, 10, 15, 20, 25\nX-axis (Countries): Australia, China, Hong Kong, India, Indonesia, Japan, Malaysia, New Zealand, Singapore, South Korea, Sri Lanka, Taiwan, Thailand, Vietnam\n\nStructured Table:\n\n| Country      | Speed (Mbps) |\n|--------------|--------------|\n| Australia    | 13           |\n| China        | 14           |\n| Hong Kong    | 3            |\n| India        | 9            |\n| Indonesia    | 19           |\n| Japan        | 17           |\n| Malaysia     | 14           |\n| New Zealand  | 11           |\n| Singapore    | 18           |\n| South Korea  | 12           |\n| Sri Lanka    | 20           |\n| Taiwan       | 7            |\n| Thailand  


Processing question 9:
{'id': 9, 'dropped': 'no', 'vis': 'VLAT_b', 'item': 'b_4', 'question: ': 'How many countries in Asia is the average internet speed slower than Thailand?', 'option:': '2 countries; 4 countries; 6 countries; 10 countries', 'correct': '10 countries'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_b.png
Question: How many countries in Asia is the average internet speed slower than Thailand?
Options: 2 countries; 4 countries; 6 countries; 10 countries
Correct answer: 10 countries

API Response: {'id': 'chatcmpl-B9MDNtMsRZvJMz7yVKhEFPSykaIyJ', 'object': 'chat.completion', 'created': 1741570045, 'model': 'gpt-4.5-preview-2025-02-27', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': "Task 1: Data Extraction and Table Creation\n\nNumerical values identified on axes:\nY-axis (Speed in Mbps): 0, 5, 10, 15, 20, 25\nX-axis (Countries): Australia, China, Hong Kong, India, Indonesia, Japan, Malaysia, New Zealand


Processing question 12:
{'id': 12, 'dropped': 'no', 'vis': 'VLAT_c', 'item': 'c_3', 'question: ': 'In which city is the cost of soda the highest?', 'option:': 'New York City; Las Vegas; Atalanta; San Francisco', 'correct': 'San Francisco'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_c.png
Question: In which city is the cost of soda the highest?
Options: New York City; Las Vegas; Atalanta; San Francisco
Correct answer: San Francisco

API Response: {'id': 'chatcmpl-B9MHNKNa71Qn62HG7Y2e8TqOouDPM', 'object': 'chat.completion', 'created': 1741570293, 'model': 'gpt-4.5-preview-2025-02-27', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Task 1: Data Extraction and Table Creation\n\nNumerical values on axes:\n- Y-axis (Cost in $): 0, 10, 20, 30, 40, 50, 60, 70, 80\n- X-axis (Cities): New York City, Las Vegas, Honolulu, Atlanta, Boston, Washington D.C., Chicago, Orlando, San Francisco, Seattle\n\nExtracted Soda Costs ($):\

...


Processing question 30:
{'id': 30, 'dropped': 'yes', 'vis': 'VLAT_g', 'item': 'g_4', 'question: ': 'More than 60 percent of males lie between the weights of 60 kg and 100 kg.', 'option:': 'True; False', 'correct': 'TRUE'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_g.png
Question: More than 60 percent of males lie between the weights of 60 kg and 100 kg.
Options: True; False
Correct answer: TRUE

API Response: {'id': 'chatcmpl-B9Mg2YsjYtcRUIu9eG2hpnb68vEQd', 'object': 'chat.completion', 'created': 1741571822, 'model': 'gpt-4.5-preview-2025-02-27', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Task 1: Data Extraction and Table Creation\n\nNumerical values identified on axes:\n\nX-axis (Height in cm): 160, 165, 170, 175, 180, 185, 190, 195, 200  \nY-axis (Weight in kg): 40, 60, 80, 100, 120, 140  \n\nNow, counting the data points explicitly from the graph:\n\n| Weight Range (kg) | Number of Males |\n|--------------


Processing question 32:
{'id': 32, 'dropped': 'no', 'vis': 'VLAT_g', 'item': 'g_6', 'question: ': 'A group of males are gathered around the height of 176 cm and the weight of 70 kg.', 'option:': 'True; False', 'correct': 'FALSE'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_g.png
Question: A group of males are gathered around the height of 176 cm and the weight of 70 kg.
Options: True; False
Correct answer: FALSE

API Response: {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

Answer: Error: You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.
Time taken: 0.52 s


Processing question 40:
{'id': 40, 'dropped': 'no', 'vis': 'VLAT_j', 'item': 'j_1', 'question: ': 'What was the number of girls named ‘Amelia’ in 2010 in the UK?', 'option:': '1500; 3800; 10100; 25300', 'correct': '10,100'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_j.png
Question: What was the number of girls named ‘Amelia’ in 2010 in the UK?
Options: 1500; 3800; 10100; 25300
Correct answer: 10,100

API Response: {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

Answer: Error: You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.
Time taken: 0.32 seconds
Corre


Processing question 48:
{'id': 48, 'dropped': 'no', 'vis': 'VLAT_h', 'item': 'h_2', 'question: ': 'Which city’s metro system has the largest number of stations?', 'option:': 'F; D; A; K', 'correct': 'F'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_h.png
Question: Which city’s metro system has the largest number of stations?
Options: F; D; A; K
Correct answer: F

API Response: {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

Answer: Error: You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.
Time taken: 0.53 seconds
Correct? N/A

Processing question 49:
{'id': 


Processing question 56:
{'id': 56, 'dropped': 'no', 'vis': 'VLAT_k', 'item': 'k_2', 'question: ': 'In which state was the unemployment rate the highest in 2015?', 'option:': 'Alaska (AK); New Mexico (NM); Florida (FL); New York (NY)', 'correct': 'Alaska (AK)'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_k.png
Question: In which state was the unemployment rate the highest in 2015?
Options: Alaska (AK); New Mexico (NM); Florida (FL); New York (NY)
Correct answer: Alaska (AK)

API Response: {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

Answer: Error: You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.ope